# Vue2 + Vue3 框架 · 第 11 天

## Vue 3 入门：Composition API 与 `<script setup>`

> 本篇依据第 11 天 Vue3 MD 笔记、PPT 和 Vue3 demo 整理。重点是从 Options API 迁移到 Composition API，并掌握 Vue 3.3 的 `defineOptions`、`defineModel`。

## 一、Vue 2 选项式 API 与 Vue 3 组合式 API

Vue 2 常把同一功能拆散在 `data`、`methods`、`computed` 和生命周期中；Vue 3 Composition API 可以按功能把状态、方法和副作用放在一起。

```vue
<!-- Options API -->
<script>
export default {
  data () { return { count: 0 } },
  methods: { add () { this.count++ } }
}
</script>
```

```vue
<!-- Composition API -->
<script setup>
import { ref } from 'vue'
const count = ref(0)
const add = () => count.value++
</script>
```

组合式 API 的优势：代码量更少、逻辑集中、功能可以抽成 composable 复用。Vue 3 仍然支持 Options API，迁移不必一次完成。

## 二、使用 create-vue 创建项目

Vue 官方推荐使用 Vite 驱动的 create-vue：

```bash
# Node.js 16+ 或更高版本
npm init vue@latest
cd vue-demo
npm install
npm run dev
```

关键文件：

```text
index.html          入口模板
src/main.js         createApp 创建应用
src/App.vue         根组件
src/components/     复用组件
vite.config.js      Vite 配置
```

Vue 3 入口：

```js
import { createApp } from 'vue'
import App from './App.vue'
createApp(App).mount('#app')
```

## 三、`setup` 与 `<script setup>`

### 3.1 `setup` 选项

`setup` 在 `beforeCreate` 之前执行，里面声明的数据和方法需要 return 才能在模板使用：

```js
export default {
  setup () {
    const message = 'hello'
    const log = () => console.log(message)
    return { message, log }
  }
}
```

### 3.2 `<script setup>` 语法糖

```vue
<script setup>
const message = 'hello'
const log = () => console.log(message)
</script>

<template>
  <p>{{ message }}</p>
  <button @click="log">打印</button>
</template>
```

不需要手写 `export default`，顶层变量默认暴露给模板。组件导入后也可直接在模板使用。

## 四、`reactive` 与 `ref`

### 4.1 `reactive`

接收对象或数组，返回一个响应式代理：

```vue
<script setup>
import { reactive } from 'vue'
const state = reactive({ msg: 'hello' })
const change = () => { state.msg = 'new message' }
</script>
```

### 4.2 `ref`

可以处理简单类型和对象，脚本中通过 `.value` 访问，模板中会自动解包：

```vue
<script setup>
import { ref } from 'vue'
const count = ref(0)
const add = () => { count.value++ }
</script>
<template><button @click="add">{{ count }}</button></template>
```

### 4.3 选择建议

`reactive` 适合一组相关对象字段；`ref` 类型支持更广、解构更安全，项目中可以优先使用 `ref`，记住脚本里的 `.value`。

## 五、Composition API 的 `computed` 与 `watch`

```vue
<script setup>
import { ref, computed, watch } from 'vue'
const count = ref(0)
const doubleCount = computed(() => count.value * 2)

watch(count, (newValue, oldValue) => {
  console.log(oldValue, '=>', newValue)
})
</script>
```

监听多个值：

```js
watch([count, name], ([newCount, newName], [oldCount, oldName]) => {
  console.log(newCount, newName)
})
```

立即执行和深度监听：

```js
watch(state, callback, { immediate: true, deep: true })
```

`ref` 包装对象时，默认监听引用变化；需要监听嵌套字段时开启 `deep`，或直接监听 `() => state.value.count`。

## 六、Composition API 生命周期

生命周期通过导入并调用函数注册：

```vue
<script setup>
import { onMounted, onUnmounted } from 'vue'

onMounted(() => {
  console.log('DOM 已挂载')
})

onMounted(() => {
  console.log('同一生命周期可以注册多个回调')
})

onUnmounted(() => {
  console.log('清理定时器和事件')
})
</script>
```

常见对应关系：`created`/`beforeCreate` 的逻辑通常直接写在 `setup` 顶层；`mounted` 对应 `onMounted`；`beforeUnmount`、`unmounted` 对应销毁阶段。

## 七、父子通信：`defineProps` 与 `defineEmits`

父组件：

```vue
<Child :count="count" @change="count = $event" />
```

子组件：

```vue
<script setup>
const props = defineProps({ count: Number })
const emit = defineEmits(['change'])
const add = () => emit('change', props.count + 1)
</script>
<template><button @click="add">{{ count }}</button></template>
```

`defineProps` 和 `defineEmits` 是 `<script setup>` 编译宏，不需要导入。props 仍然遵循单向数据流，子组件通过事件请求父组件修改。

## 八、模板引用、`defineExpose` 与跨层通信

### 8.1 模板引用

```vue
<script setup>
import { ref, onMounted } from 'vue'
const input = ref(null)
onMounted(() => input.value.focus())
</script>
<template><input ref="input"></template>
```

### 8.2 暴露子组件方法

`<script setup>` 默认不向父组件暴露内部变量，需要显式指定：

```js
defineExpose({ reset () { /* ... */ } })
```

父组件通过组件 ref 调用 `child.value.reset()`。

### 8.3 `provide/inject`

```js
// 祖先
import { provide, ref } from 'vue'
const theme = ref('dark')
provide('theme', theme)

// 后代
import { inject } from 'vue'
const theme = inject('theme')
```

提供 `ref` 才能让跨层数据保持响应式；复杂业务状态建议使用 Pinia。

## 九、Vue 3.3 新特性

### `defineOptions`

在 `<script setup>` 中定义组件名或其他 Options API 选项：

```vue
<script setup>
defineOptions({ name: 'UserPage' })
</script>
```

### `defineModel`

Vue 3 组件 `v-model` 默认对应 `modelValue` prop 和 `update:modelValue` 事件。Vue 3.3 可以用 `defineModel` 简化：

```vue
<script setup>
const model = defineModel()
</script>
<template><input v-model="model"></template>
```

如果项目版本或插件不支持，使用传统写法：

```js
const props = defineProps({ modelValue: String })
const emit = defineEmits(['update:modelValue'])
```

使用 `defineModel` 前确认 Vite 和 Vue 编译插件版本已经开启对应支持。

## 十、当天速记

- Vue 3 的 Composition API 用函数组织逻辑，`<script setup>` 减少样板代码。
- `reactive` 适合对象，`ref` 支持基本类型和对象；脚本中访问 ref 要写 `.value`。
- `computed` 负责派生值，`watch` 负责数据变化后的副作用。
- 生命周期用 `onMounted` 等函数注册，可多次注册。
- 父子通信使用 `defineProps`、`defineEmits`；模板引用使用 `ref`。
- `<script setup>` 下默认不暴露内部成员，使用 `defineExpose` 明确暴露。
- `provide/inject` 可以跨层共享，传入 ref 才能保持响应式。